# Phase 5 — how sure are we that this table is better?

Every table the harness builds gets a number, and every decision a run makes is a
**comparison** of two of them: is this table better than the one it came from.
Until now the run could put an interval around a *score* — `val_se`, bootstrapped
over the validation rows — and nothing at all around the *difference*.

That gap is not cosmetic. This notebook builds the difference instrument by hand,
in the order the argument has to be read:

1. why a difference is measured far better than either score behind it,
2. why that same trick is **wrong** on an ordered split, and what replaces it,
3. the bug the design had to be written around — the validation half is not
   stored in the order you think it is,
4. the real measurement, against a feature Kaggle has already refuted, where the
   instrument gets a partial pass and the failure is the interesting half.

Written up as `research/archive/decisions/0059`. It implements what
[0036](../../../research/archive/decisions/0036-the-pairing-asymmetry-was-not-there.md) proposed
and left unbuilt.

## 1. Two scores, and the difference between them

Start with the thing that makes any of this work. Two candidates are scored on the
**same** validation rows, so their errors are correlated — they miss the same rows
for mostly the same reasons — and

    Var(A - B) = Var(A) + Var(B) - 2*Cov(A, B)

The covariance term is not small. Build two predictors that differ a little and
watch what the pairing buys.

In [ ]:
import os

import numpy as np

rng = np.random.default_rng(0)
n = 400

y = rng.normal(10.0, 2.0, size=n)
good = y + rng.normal(0, 0.5, size=n)       # a decent predictor
worse = good + rng.normal(0, 0.15, size=n)  # the same predictor, slightly degraded

print(f"n rows: {n}")
print(f"correlation between the two predictions: {np.corrcoef(good, worse)[0, 1]:.3f}")

Now score both on the *same* bootstrap resample each time, rather than resampling
each independently. The difference of the paired scores is the quantity a run
actually decides on.

In [ ]:
from tabular_agent.metrics import compute_metric

B = 400
paired, indep_a, indep_b = [], [], []
rng = np.random.default_rng(1)
for _ in range(B):
    idx = rng.integers(0, n, n)              # ONE resample, used for both
    sa = compute_metric("rmse", y[idx], good[idx])
    sb = compute_metric("rmse", y[idx], worse[idx])
    paired.append(sb - sa)                   # rmse: lower is better, so b - a is A's gain
    indep_a.append(sa)
    indep_b.append(sb)

paired_se = np.std(paired)
independent_se = np.sqrt(np.var(indep_a) + np.var(indep_b))
print(f"sd of the paired difference : {paired_se:.5f}")
print(f"if the two were independent : {independent_se:.5f}")
print(f"ratio                       : {paired_se / independent_se:.3f}")

That ratio is the whole reason the harness now stores each table's **predictions**
and not just its score: you cannot pair two candidates after the fact unless both
prediction vectors survived.

0036's real finding is what *governs* that ratio, and it is not whether you varied
a model or a table — it is **how similar the two predictors are**. Two adjacent
tables one step apart came out at 0.18; two different libraries at 0.37. Watch it
move with similarity alone.

In [ ]:
from tabular_agent.compare import paired_difference

for label, degrade in (("nearly identical", 0.10),
                       ("one step apart  ", 0.30),
                       ("quite different ", 1.00)):
    other = good + rng.normal(0, degrade, size=n)
    out = paired_difference("rmse", y, good, other)
    corr = np.corrcoef(good, other)[0, 1]
    print(f"{label}  corr {corr:.3f}   paired_se {out['paired_se']:.5f}"
          f"   ratio {out['pairing_ratio']:.3f}")

Similar predictors, precise comparison — and the ratio climbs toward 1 as they
diverge, which is the point at which pairing stops buying anything. `compare.py`
reports `pairing_ratio` on every comparison rather than citing one run, so the
claim stays checkable on each dataset instead of resting on house-prices.

In [ ]:
out = paired_difference("rmse", y, good, worse)
for k, v in out.items():
    print(f"{k:18s} {v}")

Two properties worth poking at, because both are easy to get wrong:

In [ ]:
# Comparing a thing with itself must not manufacture uncertainty.
same = paired_difference("rmse", y, good, good)
print(f"gain against itself      : {same['gain']}")
print(f"spread against itself    : {same['paired_se']}")

# Misaligned predictions would silently answer a question nobody asked.
try:
    paired_difference("rmse", y, good, worse[:200])
except ValueError as e:
    print(f"\nrefused: {e}")

## 2. Why this is the wrong operation on an ordered split

A bootstrap over rows assumes the rows are exchangeable draws from one population.
Under `split_on` they are **not**: the validation half is a later *period*,
deliberately chosen to differ from the training rows. Resampling inside it measures
how much the difference wobbles within that period, and says nothing about the
thing that actually varies — which period it is.

That is why `val_se` is refused outright on ordered splits (`_score_on_held_out`
returns `None` for it). A too-small yardstick is worse than none.

So on ordered data the difference is recomputed on **consecutive blocks**. Here is
an effect that is real early and gone later — the shape this is built to expose.

In [ ]:
n = 1000
rng = np.random.default_rng(2)
y_t = rng.normal(10.0, 2.0, size=n)
order = np.arange(n)                      # the ordering column: day 0 .. 999

# A is much better over the first half, slightly worse over the second.
a = y_t + rng.normal(0, 0.2, size=n)
a[n // 2:] = y_t[n // 2:] + rng.normal(0, 1.4, size=n - n // 2)
b = y_t + rng.normal(0, 1.0, size=n)

rows = paired_difference("rmse", y_t, a, b)
print(f"row bootstrap : gain {rows['gain']:+.5f}  se {rows['paired_se']:.5f}"
      f"  -> gain/se {rows['gain'] / rows['paired_se']:.1f}")

A small positive gain, about one standard error from zero: *"nothing here."*

That is a bad summary of what is actually happening, and it is bad in a way worth
naming. A is much better over the first half of the window and clearly worse over
the second. The row bootstrap averages those into a near-wash and then reports the
wash as a *precise* near-wash — it has no notion of where a row sits, so a sign
change and a genuine absence of effect look identical to it.

Blocks do have that notion.

In [ ]:
from tabular_agent.compare import blockwise_difference

blocks = blockwise_difference("rmse", y_t, a, b, order, n_blocks=4)
print(f"pooled gain    {blocks['gain']:+.5f}")
print(f"block gains    {[round(g, 5) for g in blocks['block_gains']]}")
print(f"sign stable    {blocks['sign_stable']}")
print(f"blocks for A   {blocks['blocks_favouring_a']}/{blocks['n_blocks']}")

`sign_stable = False` is the finding. The pooled gain is positive and confident;
the blocks say the winner changed partway through the window.

Note what is **not** claimed: nothing here is a significance test, and no threshold
is attached to `block_sd`. There are two points on that curve and picking a number
between them is 0034's refusal to fit a constant to three observations.

## 3. The bug this had to be written around

Blocks are only meaningful if the rows are cut in their ordering column's order —
and **the validation half is not stored that way.** `RunSession` picks the
validation positions in time order and then writes them with `np.sort(val_pos)`,
which sorts the *positions*, so the file comes back in the input file's original
row order.

Rossmann's raw file is in **reverse** chronological order. Blocking on file order
there would cut five arbitrary slices and report a reassuring number.

So `blockwise_difference` takes the ordering column as a **required argument** and
sorts internally. Shuffling the rows must change nothing.

In [ ]:
shuffle = rng.permutation(n)
shuffled = blockwise_difference(
    "rmse", y_t[shuffle], a[shuffle], b[shuffle], order[shuffle], n_blocks=4
)
print(f"in order : {[round(g, 5) for g in blocks['block_gains']]}")
print(f"shuffled : {[round(g, 5) for g in shuffled['block_gains']]}")
print(f"identical: {shuffled['block_gains'] == blocks['block_gains']}")

And the mode is chosen by *passing the column*, not by a boolean flag — a flag
would let a caller say "ordered" without supplying the order, which is the one
combination that cannot be served.

In [ ]:
from tabular_agent.compare import compare_predictions

print(compare_predictions("rmse", y_t, a, b)["mode"])
print(compare_predictions("rmse", y_t, a, b, order_values=order)["mode"])

## 4. Through the harness, on a run

Nothing above touched a run. In the harness the instrument rides along with the
ruler: every table is probed on creation, its predictions on the validation half
are stored beside it, and the comparison against its parent gets an interval
automatically.

In [ ]:
import pandas as pd
from tabular_agent.harness.session import RunSession
from tabular_agent.harness.tools import inspect_data, transform_features
from tabular_agent.taskspec import TaskSpec

tmp = os.path.join(os.getcwd(), "_nb_interval")
os.makedirs(tmp, exist_ok=True)

rng = np.random.default_rng(0)
m = 1200
x, z = rng.normal(size=m), rng.normal(size=m)
df = pd.DataFrame({
    "x": x, "z": z, "day": np.arange(m),
    "y": 5.0 * x + 3.0 * z + rng.normal(scale=0.5, size=m) + 20.0,
})
data_path = os.path.join(tmp, "data.parquet")
df.to_parquet(data_path)

session = RunSession(
    run_dir=os.path.join(tmp, "run"),
    data_path=data_path,
    spec=TaskSpec(target="y", task_type="regression", metric="rmse", seed=0),
)
inspect_data(session)
print("split:", "random")

In [ ]:
out = transform_features(
    session,
    "def transform_features(df):\n"
    "    df['signal'] = df['x'] * 2.0 + df['z']\n"
    "    return df\n",
)
print(f"probe score          {out['probe_score']}")
print(f"vs source table      {out['probe_vs_source_table']:+.5f}")
print("interval:")
for k, v in out["probe_vs_source_interval"].items():
    print(f"  {k:18s} {v}")

The gain now arrives with the spread of that same gain beside it, in the metric's
own units. That is the number 0036 asked for and nothing had ever reported.

The sidecar it depends on is one vector per table:

In [ ]:
from tabular_agent.harness.tools import _probe_preds_path, _load_probe_preds

p = _probe_preds_path(session, os.path.join(session.run_dir, out["table"]))
preds = _load_probe_preds(session, os.path.join(session.run_dir, out["table"]))
print(f"stored at : {os.path.relpath(p, session.run_dir)}")
print(f"length    : {len(preds)}  (= validation rows)")
print(f"bytes     : {os.path.getsize(p):,}")

And it degrades rather than crashes. Delete the parent's sidecar and the score
survives; only the interval goes.

In [ ]:
os.remove(_probe_preds_path(session, session.data_path))
degraded = transform_features(
    session,
    "def transform_features(df):\n"
    "    df['noise'] = df['x'] * 0.0\n"
    "    return df\n",
)
print(f"probe score still there : {degraded['probe_score']}")
print(f"interval present        : {'probe_vs_source_interval' in degraded}")

## 5. The real measurement, and the half that failed

The synthetic case above was built to be caught. The honest test is a case whose
answer is already known from outside, and this project has exactly one.

`_runs/ship_rossmann_median` ends its chain with `t9`, a `fit_history` table built
on `t8`. 0056 scored that arm on Kaggle's private rows: **0.18438** with history
against **0.15429** without — the feature **lost 0.0301**. Our own validation block
ranked it best of four arms.

`scripts/probe_difference_interval.py` runs the instrument against it. Predictions
were registered in its docstring before it ran. The numbers:

| instrument | reading |
|---|---|
| pooled gain, t9 over t8 | **+0.02028** |
| paired bootstrap over rows | se **0.00074** → gain/se **27.3** |
| pairing ratio | **0.371** |
| blocks, 44-day validation window | sd **0.01214**, `sign_stable` **True**, 5/5 |
| blocks, 90-day window (val + holdout) | sd **0.01670**, `sign_stable` **False**, 8/10 |
| Kaggle, Aug-Sep | **−0.0301** |

Read those in order, because the story is in the sequence.

**The row bootstrap is not merely imprecise here, it is confidently wrong.**
Twenty-seven standard errors from zero, for a difference that reverses on unseen
rows.

**0036's pairing ratio replicated.** 0.371 lands inside its 0.18-0.37, on a second
dataset, a different metric, and 50,175 validation rows instead of 292. That was
the thing 0036 said it could not establish.

**Blocking helped 16-fold and still was not enough.** Over the validation half the
interval widens from 0.00074 to 0.01214 — `gain/se` falls from 27.3 to 1.7, which
is the honest reading — but all five blocks still favour the history table.
**A run would not have caught this feature.**

**The window was the problem, not the block count.** Doubled to 90 days, the blocks
disagree, the pooled gain nearly halves, and the decay is visible directly: the
first five blocks average **+0.0200**, the last five **+0.0022**. Kaggle's test
window starts two weeks after the last block and reads −0.0301 — the same line,
continued.

So the limitation `compare.py` states in its own docstring, before any of this was
measured, is the one that bit: **blocking samples the periods it has.** If an
effect changes sign beyond the far edge of the evaluation window, every block can
agree and the effect can still fail on the rows that matter. More blocks over the
same 44 days would not have helped. Reaching further out would — and that is a
fact about the exam (0043), not about this instrument.

### What was deliberately not done

`min_gain_ratio` is **unchanged** at 0.25 of `probe_se`. 0028 left its magnitude
open and 0036 proposed this instrument as a candidate answer, but the measurement
above is one comparison on one dataset. Choosing a multiple of the block sd from it
would be fitting a constant to a single observation — which is exactly the move
0034 tested and rejected.

The interval is reported. Nothing is gated on it yet.

In [ ]:
import shutil
shutil.rmtree(tmp, ignore_errors=True)
print("cleaned up")